# Question 4 — Web Scraping to Extract GameStop Revenue Data

Scrape GameStop's annual revenue history from Stock Analysis, clean the values, and display the last five rows. Revenue values are normalized to **USD millions** for consistency with the graph.

In [ ]:
%pip install -q yfinance pandas requests beautifulsoup4 lxml plotly

In [ ]:
import re
from io import StringIO

import pandas as pd
import requests
import yfinance as yf
from bs4 import BeautifulSoup
from IPython.display import display


## Define the web-scraping function

In [ ]:
def scrape_annual_revenue(ticker: str) -> pd.DataFrame:
    """Scrape annual revenue from Stock Analysis. Revenue is returned in USD millions."""
    symbol = ticker.strip().lower()
    url = f"https://stockanalysis.com/stocks/{symbol}/revenue/"
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        )
    }
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()

    try:
        tables = pd.read_html(StringIO(response.text))
    except ValueError as exc:
        raise RuntimeError(
            f"No readable HTML table was found on {url}. The page may have changed "
            "its layout or may be rate-limiting automated requests."
        ) from exc

    revenue_table = None
    for table in tables:
        if isinstance(table.columns, pd.MultiIndex):
            table.columns = [
                " ".join(str(x) for x in col if str(x) != "nan").strip()
                for col in table.columns
            ]
        columns = list(table.columns)
        date_col = next(
            (c for c in columns if "fiscal year" in str(c).lower()
             or "period end" in str(c).lower()
             or str(c).strip().lower() in {"date", "year"}),
            None,
        )
        revenue_col = next(
            (c for c in columns if "revenue" in str(c).lower()), None
        )
        if date_col is not None and revenue_col is not None:
            revenue_table = table[[date_col, revenue_col]].copy()
            break

    if revenue_table is None:
        raise RuntimeError(
            f"Could not find Date/Year and Revenue columns on {url}. "
            "Inspect pd.read_html(StringIO(response.text)) and update the scraper."
        )

    revenue_table.columns = ["Date", "Revenue"]
    revenue_table["Date"] = pd.to_datetime(revenue_table["Date"], errors="coerce")

    def to_usd_millions(value):
        if pd.isna(value):
            return float("nan")
        text = str(value).strip().replace("$", "").replace(",", "")
        if text in {"", "-", "—", "nan", "None"}:
            return float("nan")
        suffix = text[-1].upper() if text[-1].isalpha() else ""
        number_text = re.sub(r"[^0-9.\-]", "", text)
        if not number_text:
            return float("nan")
        number = float(number_text)
        if suffix == "B":
            number *= 1000  # billions to millions
        elif suffix == "K":
            number /= 1000  # thousands to millions
        return number  # M or plain values are already in millions

    revenue_table["Revenue"] = revenue_table["Revenue"].apply(to_usd_millions)
    revenue_table = revenue_table.dropna(subset=["Date", "Revenue"])
    revenue_table = revenue_table.sort_values("Date").reset_index(drop=True)
    if revenue_table.empty:
        raise RuntimeError(f"No usable annual revenue rows were found for {ticker.upper()}.")
    return revenue_table


## Scrape GME revenue and display the last five rows

In [ ]:
gme_revenue = scrape_annual_revenue("GME")
display(gme_revenue.tail())

## Save the scraped data

In [ ]:
gme_revenue.to_csv("gme_revenue.csv", index=False)
print("Saved gme_revenue.csv")

## Submission screenshot

Capture the executed `gme_revenue.tail()` cell and its five-row output. Source page: https://stockanalysis.com/stocks/gme/revenue/. If the site's HTML changes or blocks a request, inspect its tables and update the column selection in `scrape_annual_revenue()`.